# Multi-Model TCN Autoencoder: Training and Comprehensive Comparison

## 📋 Overview
This notebook trains and compares **Temporal Convolutional Network (TCN) Autoencoders** with different latent space dimensions for spectral data compression and representation learning.

### 🎯 Objectives
1. Train TCN autoencoders with **three latent dimensions**: 128, 64, and 32
2. Evaluate each model using multiple metrics
3. Compare performance across all models
4. Visualize latent space representations

### 📊 Evaluation Metrics
- **Reconstruction Error**: MSE between input and reconstructed spectra
- **Nearest Neighbor Similarity**: Preservation of spectral similarity in latent space
- **Silhouette Score**: Quality of class clustering in latent space
- **F1 Score**: Classification accuracy using latent representations
- **t-SNE Visualization**: 2D projection of latent space

### 🔧 Architecture
The TCN autoencoder uses:
- Temporal convolutional blocks with residual connections
- Progressive downsampling in encoder (1000 → latent_dim)
- Progressive upsampling in decoder (latent_dim → 1000)
- Batch normalization and dropout for regularization

---

## 1. Environment Setup and Imports

In [ ]:
# Core libraries
import os
import datetime
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import savgol_filter

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Scikit-learn
from sklearn.neighbors import NearestNeighbors, KNeighborsClassifier
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import f1_score, silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.manifold import TSNE

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Configure matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ All libraries imported successfully")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Configuration

In [ ]:
# Configuration
CONFIG = {
    # Model configurations
    'latent_dims': [128, 64, 32],
    'input_length': 1000,
    
    # Training parameters
    'epochs': 50,
    'batch_size': 128,
    'learning_rate': 0.001,
    
    # Data
    'data_path': 'cda_qm_spectra_pre2008277_train_lvl2.parquet',
    
    # Output directories
    'output_dir': './outputs',
    'comparison_dir': './comparison_results',
}

# Create output directories
os.makedirs(CONFIG['output_dir'], exist_ok=True)
os.makedirs(CONFIG['comparison_dir'], exist_ok=True)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"\nConfiguration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

## 3. Model Architecture

### Temporal Convolutional Block
Each block contains:
- Two convolutional layers with batch normalization
- Residual connection
- ReLU activation and dropout

In [ ]:
class TemporalConvBlock(nn.Module):
    """
    Temporal Convolutional Block with residual connection
    
    Args:
        in_channels: Number of input channels
        out_channels: Number of output channels
        kernel_size: Size of convolutional kernel
        stride: Stride for convolution
        dilation: Dilation rate
        dropout: Dropout probability
    """
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, dilation=1, dropout=0.2):
        super(TemporalConvBlock, self).__init__()
        padding = dilation * (kernel_size - 1) // 2
        
        # First convolutional layer
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size,
                              stride=stride, padding=padding, dilation=dilation)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)
        
        # Second convolutional layer
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size,
                              stride=1, padding=padding, dilation=dilation)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)
        
        # Residual connection
        self.residual = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.residual = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1, stride=stride),
                nn.BatchNorm1d(out_channels)
            )
    
    def forward(self, x):
        residual = self.residual(x)
        
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu1(out)
        out = self.dropout1(out)
        
        out = self.conv2(out)
        out = self.bn2(out)
        out = out + residual  # Add residual
        out = self.relu2(out)
        out = self.dropout2(out)
        
        return out

print("✓ TemporalConvBlock defined")

In [ ]:
class TCN_Autoencoder(nn.Module):
    """
    Temporal Convolutional Network Autoencoder
    
    Architecture:
        Encoder: 1000 → 500 → 250 → 125 → 62 → 31 → latent_dim
        Decoder: latent_dim → 31 → 62 → 125 → 250 → 500 → 1000
    
    Args:
        input_len: Length of input sequence (default: 1000)
        latent_dim: Dimension of latent representation
    """
    def __init__(self, input_len=1000, latent_dim=128):
        super(TCN_Autoencoder, self).__init__()
        
        self.input_len = input_len
        self.latent_dim = latent_dim
        
        # ======================== ENCODER ========================
        # Progressive downsampling: 1000 → 500 → 250 → 125 → 62 → 31
        self.enc_block1 = TemporalConvBlock(1, 32, kernel_size=5, stride=2, dilation=1)
        self.enc_block2 = TemporalConvBlock(32, 64, kernel_size=5, stride=2, dilation=2)
        self.enc_block3 = TemporalConvBlock(64, 128, kernel_size=5, stride=2, dilation=4)
        self.enc_block4 = TemporalConvBlock(128, 256, kernel_size=5, stride=2, dilation=8)
        self.enc_block5 = TemporalConvBlock(256, 256, kernel_size=5, stride=2, dilation=16)
        
        # Adaptive pooling and latent projection
        self.adaptive_pool = nn.AdaptiveAvgPool1d(1)
        self.fc_latent = nn.Linear(256, latent_dim)
        
        # ======================== DECODER ========================
        # Expand from latent space
        self.fc_expand = nn.Linear(latent_dim, 256 * 32)
        
        # Progressive upsampling: 32 → 64 → 128 → 256 → 512 → 1000
        self.dec_block1 = nn.Sequential(
            nn.ConvTranspose1d(256, 256, kernel_size=5, stride=2, padding=2, output_padding=0),
            nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.2)
        )
        self.dec_block2 = nn.Sequential(
            nn.ConvTranspose1d(256, 128, kernel_size=5, stride=2, padding=2, output_padding=0),
            nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.2)
        )
        self.dec_block3 = nn.Sequential(
            nn.ConvTranspose1d(128, 64, kernel_size=5, stride=2, padding=2, output_padding=1),
            nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.2)
        )
        self.dec_block4 = nn.Sequential(
            nn.ConvTranspose1d(64, 32, kernel_size=5, stride=2, padding=2, output_padding=1),
            nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(0.2)
        )
        self.dec_block5 = nn.Sequential(
            nn.ConvTranspose1d(32, 1, kernel_size=5, stride=2, padding=2, output_padding=1),
            nn.Sigmoid()  # Output range [0, 1]
        )
    
    def encode(self, x):
        """Encode input to latent representation"""
        x = self.enc_block1(x)
        x = self.enc_block2(x)
        x = self.enc_block3(x)
        x = self.enc_block4(x)
        x = self.enc_block5(x)
        x = self.adaptive_pool(x)
        x = x.squeeze(-1)
        latent = self.fc_latent(x)
        return latent
    
    def decode(self, latent):
        """Decode latent representation to reconstruction"""
        x = self.fc_expand(latent)
        x = x.view(-1, 256, 32)
        x = self.dec_block1(x)
        x = self.dec_block2(x)
        x = self.dec_block3(x)
        x = self.dec_block4(x)
        x = self.dec_block5(x)
        return x
    
    def forward(self, x):
        """Full forward pass: encode → decode"""
        latent = self.encode(x)
        reconstruction = self.decode(latent)
        return reconstruction, latent

print("✓ TCN_Autoencoder defined")

# Test model instantiation
test_model = TCN_Autoencoder(input_len=1000, latent_dim=128)
total_params = sum(p.numel() for p in test_model.parameters())
print(f"✓ Test model created: {total_params:,} parameters")

## 4. Data Loading and Preprocessing

### Preprocessing Pipeline
1. Truncate or pad to target length (1000)
2. Apply Savitzky-Golay filter for smoothing
3. Log transformation: log(1 + x)
4. Min-max normalization

In [ ]:
def preprocess_spectra(spectra, target_length=1000):
    """
    Preprocess spectral data
    
    Args:
        spectra: Array of spectra
        target_length: Target sequence length
    
    Returns:
        Preprocessed spectra array
    """
    processed = []
    
    for spec in spectra:
        # 1. Truncate or pad to target length
        if len(spec) >= target_length:
            s = spec[:target_length]
        else:
            s = np.pad(spec, (0, target_length - len(spec)), 'constant')
        
        # 2. Savitzky-Golay smoothing
        window_length = min(11, len(s) if len(s) % 2 == 1 else len(s) - 1)
        if window_length >= 5:
            s = savgol_filter(s, window_length=window_length, polyorder=3)
        
        # 3. Log transformation
        s = np.log1p(np.maximum(s, 0))
        
        # 4. Min-max normalization
        max_val = np.max(s)
        if max_val > 0:
            s = s / max_val
            
        processed.append(s)
    
    return np.array(processed, dtype=np.float32)

print("✓ Preprocessing function defined")

In [ ]:
# Load data
print("="*70)
print("LOADING AND PREPROCESSING DATA")
print("="*70)

df = pd.read_parquet(CONFIG['data_path'])
print(f"✓ Loaded data: {df.shape}")
print(f"  Columns: {list(df.columns)}")

# Extract spectra
spectra_raw = np.stack(df['spectrum'].values)
print(f"✓ Raw spectra shape: {spectra_raw.shape}")

# Preprocess
X_train = preprocess_spectra(spectra_raw, target_length=CONFIG['input_length'])
print(f"✓ Preprocessed shape: {X_train.shape}")
print(f"  Value range: [{X_train.min():.4f}, {X_train.max():.4f}]")

# Extract class labels if available
if 'class' in df.columns:
    class_labels = df['class'].values
    unique_classes = np.unique(class_labels)
    class_to_idx = {cls: idx for idx, cls in enumerate(unique_classes)}
    numeric_labels = np.array([class_to_idx[cls] for cls in class_labels])
    print(f"✓ Class labels: {len(unique_classes)} unique classes")
    HAS_LABELS = True
else:
    class_labels = None
    numeric_labels = None
    HAS_LABELS = False
    print("⚠ No class labels found")

print("="*70)

In [ ]:
# Visualize sample spectra
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Sample Preprocessed Spectra', fontsize=16, fontweight='bold')

for i, ax in enumerate(axes.flat):
    ax.plot(X_train[i], linewidth=0.8, alpha=0.8)
    ax.set_title(f'Spectrum {i+1}' + (f' (Class: {class_labels[i]})' if HAS_LABELS else ''))
    ax.set_xlabel('Index')
    ax.set_ylabel('Normalized Intensity')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Displayed 6 sample spectra")

## 5. Model Training

### Training Strategy
- **Loss Function**: Mean Squared Error (MSE) for reconstruction
- **Optimizer**: Adam with learning rate 0.001
- **Class Weights**: Applied to handle class imbalance
- **Epochs**: 50 per model
- **Models**: Train 3 models with latent_dim = 128, 64, 32

In [ ]:
def compute_class_weights(labels, num_samples):
    """
    Compute class weights for balanced training
    
    Args:
        labels: Class labels
        num_samples: Total number of samples
    
    Returns:
        Tensor of sample weights
    """
    if labels is None:
        return torch.ones(num_samples)
    
    unique_classes, class_counts = np.unique(labels, return_counts=True)
    class_weights = 1.0 / class_counts
    class_weights = class_weights / class_weights.sum() * len(unique_classes)
    
    label_to_weight = {label: weight for label, weight in zip(unique_classes, class_weights)}
    sample_weights = np.array([label_to_weight[label] for label in labels])
    
    return torch.from_numpy(sample_weights).float()

# Compute sample weights
sample_weights = compute_class_weights(numeric_labels, len(X_train))
print(f"✓ Sample weights computed")
if HAS_LABELS:
    print(f"  Weight range: [{sample_weights.min():.4f}, {sample_weights.max():.4f}]")

In [ ]:
def train_model(model, train_loader, sample_weights, epochs, learning_rate, device, latent_dim):
    """
    Train TCN Autoencoder model
    
    Args:
        model: TCN_Autoencoder model
        train_loader: DataLoader for training data
        sample_weights: Sample weights for balanced training
        epochs: Number of training epochs
        learning_rate: Learning rate
        device: Training device (CPU/GPU)
        latent_dim: Latent dimension (for logging)
    
    Returns:
        Dictionary containing training history and model
    """
    model = model.to(device)
    criterion = nn.MSELoss(reduction='none')
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    history = {'train_loss': [], 'epoch_times': []}
    
    print(f"\n{'='*70}")
    print(f"TRAINING MODEL: Latent Dim = {latent_dim}")
    print(f"{'='*70}")
    
    start_time = datetime.datetime.now()
    
    for epoch in range(epochs):
        epoch_start = datetime.datetime.now()
        model.train()
        epoch_loss = 0.0
        batch_count = 0
        
        for batch_idx, (batch,) in enumerate(train_loader):
            batch = batch.to(device)
            batch_indices = range(batch_idx * train_loader.batch_size, 
                                 min((batch_idx + 1) * train_loader.batch_size, len(sample_weights)))
            weights = sample_weights[batch_indices].to(device)
            
            # Forward pass
            reconstruction, latent = model(batch)
            
            # Compute weighted loss
            loss_per_sample = criterion(reconstruction, batch).mean(dim=[1, 2])
            loss = (loss_per_sample * weights).mean()
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            batch_count += 1
        
        avg_loss = epoch_loss / batch_count
        history['train_loss'].append(avg_loss)
        
        epoch_time = (datetime.datetime.now() - epoch_start).total_seconds()
        history['epoch_times'].append(epoch_time)
        
        # Print progress every 10 epochs
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoch [{epoch+1:3d}/{epochs}]  Loss: {avg_loss:.6f}  Time: {epoch_time:.2f}s")
    
    total_time = (datetime.datetime.now() - start_time).total_seconds()
    print(f"\n✓ Training complete in {total_time:.2f}s ({total_time/60:.2f} minutes)")
    print(f"  Final loss: {history['train_loss'][-1]:.6f}")
    
    return {
        'model': model,
        'history': history,
        'config': {
            'latent_dim': latent_dim,
            'epochs': epochs,
            'learning_rate': learning_rate,
            'total_time': total_time
        }
    }

print("✓ Training function defined")

### Train All Models

Training 3 models sequentially with latent dimensions: 128, 64, 32

In [ ]:
# Prepare data loader
X_tensor = torch.from_numpy(X_train).unsqueeze(1)  # Shape: (N, 1, 1000)
dataset = TensorDataset(X_tensor)
train_loader = DataLoader(dataset, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=0)

print(f"✓ DataLoader created: {len(train_loader)} batches")

In [ ]:
# Train all models
trained_models = {}

print("\n" + "="*70)
print("TRAINING ALL MODELS")
print("="*70)
print(f"Latent dimensions to train: {CONFIG['latent_dims']}")
print(f"Epochs per model: {CONFIG['epochs']}")
print(f"Total training runs: {len(CONFIG['latent_dims'])}")
print("="*70)

overall_start = datetime.datetime.now()

for latent_dim in CONFIG['latent_dims']:
    # Create model
    model = TCN_Autoencoder(input_len=CONFIG['input_length'], latent_dim=latent_dim)
    
    # Train model
    result = train_model(
        model=model,
        train_loader=train_loader,
        sample_weights=sample_weights,
        epochs=CONFIG['epochs'],
        learning_rate=CONFIG['learning_rate'],
        device=device,
        latent_dim=latent_dim
    )
    
    # Save model
    model_path = os.path.join(CONFIG['output_dir'], 
                             f"tcn_autoencoder_latent{latent_dim}_epoch{CONFIG['epochs']}_latest.pth")
    
    torch.save({
        'model_state_dict': result['model'].state_dict(),
        'config': result['config'],
        'history': result['history']
    }, model_path)
    
    print(f"✓ Model saved: {model_path}")
    
    # Store result
    trained_models[latent_dim] = result

overall_time = (datetime.datetime.now() - overall_start).total_seconds()
print(f"\n{'='*70}")
print(f"ALL MODELS TRAINED SUCCESSFULLY")
print(f"{'='*70}")
print(f"Total training time: {overall_time:.2f}s ({overall_time/60:.2f} minutes)")
print(f"Models trained: {list(trained_models.keys())}")
print(f"{'='*70}")

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Training Progress for All Models', fontsize=16, fontweight='bold')

colors = ['#2E86AB', '#A23B72', '#F18F01']

# Loss curves
for idx, (latent_dim, result) in enumerate(trained_models.items()):
    axes[0].plot(result['history']['train_loss'], 
                label=f'Latent Dim = {latent_dim}',
                color=colors[idx], linewidth=2, alpha=0.8)

axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('MSE Loss', fontsize=12)
axes[0].set_title('Training Loss', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_yscale('log')

# Training time per epoch
for idx, (latent_dim, result) in enumerate(trained_models.items()):
    axes[1].plot(result['history']['epoch_times'], 
                label=f'Latent Dim = {latent_dim}',
                color=colors[idx], linewidth=2, alpha=0.8)

axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Time (seconds)', fontsize=12)
axes[1].set_title('Training Time per Epoch', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Training curves displayed")

## 6. Model Evaluation

### Evaluation Metrics

1. **Reconstruction Error**: Mean Squared Error between input and output
2. **Nearest Neighbor Similarity**: How well spectral similarity is preserved in latent space
3. **Silhouette Score**: Quality of clustering by class in latent space
4. **F1 Score**: Classification performance using latent representations
5. **t-SNE Visualization**: 2D projection of latent space

In [ ]:
def evaluate_model(model, data_loader, X_original, class_labels, numeric_labels, device):
    """
    Comprehensive model evaluation
    
    Args:
        model: Trained TCN_Autoencoder
        data_loader: DataLoader for evaluation
        X_original: Original preprocessed data
        class_labels: String class labels
        numeric_labels: Numeric class labels
        device: Evaluation device
    
    Returns:
        Dictionary containing all evaluation metrics
    """
    model.eval()
    
    # Extract latent representations and reconstructions
    latent_vectors = []
    reconstructions_list = []
    reconstruction_errors = []
    
    with torch.no_grad():
        for (batch,) in data_loader:
            batch = batch.to(device)
            recon, latent = model(batch)
            
            # Compute reconstruction error per sample
            batch_errors = torch.mean((recon - batch) ** 2, dim=[1, 2]).cpu().numpy()
            
            latent_vectors.append(latent.cpu().numpy())
            reconstructions_list.append(recon.cpu().numpy())
            reconstruction_errors.append(batch_errors)
    
    latent_vectors = np.concatenate(latent_vectors, axis=0)
    reconstructions_array = np.concatenate(reconstructions_list, axis=0)
    reconstruction_errors = np.concatenate(reconstruction_errors, axis=0)
    
    results = {
        'latent_vectors': latent_vectors,
        'reconstructions': reconstructions_array,
        'reconstruction_errors': reconstruction_errors,
        'mean_reconstruction_error': np.mean(reconstruction_errors),
        'std_reconstruction_error': np.std(reconstruction_errors)
    }
    
    # 1. Nearest Neighbor Evaluation
    print("  Computing nearest neighbor similarity...")
    k = 10
    nbrs = NearestNeighbors(n_neighbors=k+1, metric='euclidean')
    nbrs.fit(latent_vectors)
    _, indices = nbrs.kneighbors(latent_vectors)
    
    # Compute spectral similarity in original space
    spectral_similarity = cosine_similarity(X_original)
    preservation_scores = []
    
    for i in range(len(X_original)):
        neighbor_indices = indices[i, 1:]  # Exclude self
        neighbor_similarities = spectral_similarity[i, neighbor_indices]
        preservation_scores.append(np.mean(neighbor_similarities))
    
    preservation_scores = np.array(preservation_scores)
    results['preservation_scores'] = preservation_scores
    results['nn_similarity'] = np.mean(preservation_scores)
    results['nn_above_threshold'] = np.mean(preservation_scores > 0.7) * 100
    
    # 2. Silhouette Score
    if numeric_labels is not None:
        print("  Computing silhouette score...")
        n_samples = min(5000, len(latent_vectors))
        indices_subset = np.random.choice(len(latent_vectors), n_samples, replace=False)
        
        silhouette = silhouette_score(
            latent_vectors[indices_subset],
            numeric_labels[indices_subset],
            metric='euclidean'
        )
        results['silhouette'] = silhouette
    else:
        results['silhouette'] = None
    
    # 3. F1 Score (KNN classification in latent space)
    if numeric_labels is not None:
        print("  Computing F1 score...")
        X_train_split, X_test_split, y_train_split, y_test_split = train_test_split(
            latent_vectors, numeric_labels, test_size=0.2, random_state=42, stratify=numeric_labels
        )
        
        knn = KNeighborsClassifier(n_neighbors=5)
        knn.fit(X_train_split, y_train_split)
        y_pred = knn.predict(X_test_split)
        
        results['f1_weighted'] = f1_score(y_test_split, y_pred, average='weighted')
        results['f1_macro'] = f1_score(y_test_split, y_pred, average='macro')
    else:
        results['f1_weighted'] = None
        results['f1_macro'] = None
    
    # 4. t-SNE Visualization
    print("  Computing t-SNE projection...")
    tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
    results['tsne_2d'] = tsne.fit_transform(latent_vectors)
    
    return results

print("✓ Evaluation function defined")

In [ ]:
# Evaluate all models
print("\n" + "="*70)
print("EVALUATING ALL MODELS")
print("="*70)

# Create evaluation data loader (no shuffling)
eval_loader = DataLoader(dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=0)

evaluation_results = {}

for latent_dim in CONFIG['latent_dims']:
    print(f"\nLatent Dim = {latent_dim}")
    print("-" * 70)
    
    model = trained_models[latent_dim]['model']
    
    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print(f"  Parameters: {total_params:,} (trainable: {trainable_params:,})")
    print(f"  Compression ratio: {CONFIG['input_length'] / latent_dim:.2f}x")
    
    # Evaluate
    eval_results = evaluate_model(
        model=model,
        data_loader=eval_loader,
        X_original=X_train,
        class_labels=class_labels,
        numeric_labels=numeric_labels,
        device=device
    )
    
    # Add model info
    eval_results['total_params'] = total_params
    eval_results['trainable_params'] = trainable_params
    
    # Store results
    evaluation_results[latent_dim] = eval_results
    
    # Print summary
    print(f"\n  Results:")
    print(f"    Reconstruction Error: {eval_results['mean_reconstruction_error']:.6f} ± {eval_results['std_reconstruction_error']:.6f}")
    print(f"    NN Similarity: {eval_results['nn_similarity']:.4f}")
    print(f"    NN Above 0.7: {eval_results['nn_above_threshold']:.2f}%")
    if eval_results['silhouette'] is not None:
        print(f"    Silhouette Score: {eval_results['silhouette']:.4f}")
    if eval_results['f1_weighted'] is not None:
        print(f"    F1 Score (weighted): {eval_results['f1_weighted']:.4f}")
        print(f"    F1 Score (macro): {eval_results['f1_macro']:.4f}")

print("\n" + "="*70)
print("EVALUATION COMPLETE")
print("="*70)

## 7. Comprehensive Model Comparison

In [ ]:
# Comparison Table
fig, ax = plt.subplots(figsize=(16, 4))
ax.axis('off')

table_data = []
headers = ['Latent\nDim', 'Parameters', 'Compression\nRatio', 'Recon Error\n(×10⁻³)', 
           'NN\nSimilarity', 'NN>0.7\n(%)', 'Silhouette', 'F1\n(weighted)', 'F1\n(macro)']

for dim in CONFIG['latent_dims']:
    r = evaluation_results[dim]
    table_data.append([
        dim,
        f"{r['total_params']:,}",
        f"{CONFIG['input_length'] / dim:.1f}x",
        f"{r['mean_reconstruction_error']*1000:.3f}",
        f"{r['nn_similarity']:.4f}",
        f"{r['nn_above_threshold']:.1f}",
        f"{r['silhouette']:.4f}" if r['silhouette'] is not None else 'N/A',
        f"{r['f1_weighted']:.4f}" if r['f1_weighted'] is not None else 'N/A',
        f"{r['f1_macro']:.4f}" if r['f1_macro'] is not None else 'N/A',
    ])

table = ax.table(cellText=table_data, colLabels=headers, cellLoc='center', loc='center',
                colWidths=[0.08, 0.13, 0.11, 0.13, 0.11, 0.09, 0.11, 0.12, 0.10])
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 2.5)

# Style header
for i in range(len(headers)):
    table[(0, i)].set_facecolor('#4CAF50')
    table[(0, i)].set_text_props(weight='bold', color='white')

# Alternate row colors
for i in range(1, len(table_data) + 1):
    for j in range(len(headers)):
        if i % 2 == 0:
            table[(i, j)].set_facecolor('#f0f0f0')

plt.title('Model Comparison: Performance Metrics Summary', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['comparison_dir'], 'comparison_table.png'), 
            dpi=150, bbox_inches='tight')
plt.show()

print("✓ Comparison table generated")

In [ ]:
# Metrics Comparison Bar Charts
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Model Comparison: All Metrics', fontsize=18, fontweight='bold')

dims = CONFIG['latent_dims']
colors = ['#2E86AB', '#A23B72', '#F18F01']

# 1. Reconstruction Error
axes[0, 0].bar(range(len(dims)), 
              [evaluation_results[d]['mean_reconstruction_error'] for d in dims], 
              color=colors, edgecolor='black', linewidth=1.5)
axes[0, 0].set_xticks(range(len(dims)))
axes[0, 0].set_xticklabels(dims)
axes[0, 0].set_ylabel('Mean Squared Error', fontsize=11)
axes[0, 0].set_title('Reconstruction Error', fontsize=13, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3, axis='y')

# 2. NN Similarity
axes[0, 1].bar(range(len(dims)), 
              [evaluation_results[d]['nn_similarity'] for d in dims], 
              color=colors, edgecolor='black', linewidth=1.5)
axes[0, 1].axhline(0.7, color='red', linestyle='--', linewidth=2, alpha=0.7, label='Threshold (0.7)')
axes[0, 1].set_xticks(range(len(dims)))
axes[0, 1].set_xticklabels(dims)
axes[0, 1].set_ylabel('Cosine Similarity', fontsize=11)
axes[0, 1].set_title('Nearest Neighbor Similarity', fontsize=13, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3, axis='y')

# 3. NN Above Threshold
axes[0, 2].bar(range(len(dims)), 
              [evaluation_results[d]['nn_above_threshold'] for d in dims], 
              color=colors, edgecolor='black', linewidth=1.5)
axes[0, 2].set_xticks(range(len(dims)))
axes[0, 2].set_xticklabels(dims)
axes[0, 2].set_ylabel('Percentage (%)', fontsize=11)
axes[0, 2].set_title('Neighbors Above 0.7 Threshold', fontsize=13, fontweight='bold')
axes[0, 2].grid(True, alpha=0.3, axis='y')

# 4. Silhouette Score
if all(evaluation_results[d]['silhouette'] is not None for d in dims):
    axes[1, 0].bar(range(len(dims)), 
                  [evaluation_results[d]['silhouette'] for d in dims], 
                  color=colors, edgecolor='black', linewidth=1.5)
    axes[1, 0].set_xticks(range(len(dims)))
    axes[1, 0].set_xticklabels(dims)
    axes[1, 0].set_ylabel('Score', fontsize=11)
    axes[1, 0].set_title('Silhouette Score (Clustering Quality)', fontsize=13, fontweight='bold')
    axes[1, 0].grid(True, alpha=0.3, axis='y')
else:
    axes[1, 0].text(0.5, 0.5, 'No class labels available', 
                   ha='center', va='center', fontsize=14, 
                   transform=axes[1, 0].transAxes)
    axes[1, 0].set_title('Silhouette Score', fontsize=13, fontweight='bold')

# 5. F1 Weighted
if all(evaluation_results[d]['f1_weighted'] is not None for d in dims):
    axes[1, 1].bar(range(len(dims)), 
                  [evaluation_results[d]['f1_weighted'] for d in dims], 
                  color=colors, edgecolor='black', linewidth=1.5)
    axes[1, 1].set_xticks(range(len(dims)))
    axes[1, 1].set_xticklabels(dims)
    axes[1, 1].set_ylabel('F1 Score', fontsize=11)
    axes[1, 1].set_title('F1 Score (Weighted)', fontsize=13, fontweight='bold')
    axes[1, 1].grid(True, alpha=0.3, axis='y')
else:
    axes[1, 1].text(0.5, 0.5, 'No class labels available', 
                   ha='center', va='center', fontsize=14,
                   transform=axes[1, 1].transAxes)
    axes[1, 1].set_title('F1 Score (Weighted)', fontsize=13, fontweight='bold')

# 6. F1 Macro
if all(evaluation_results[d]['f1_macro'] is not None for d in dims):
    axes[1, 2].bar(range(len(dims)), 
                  [evaluation_results[d]['f1_macro'] for d in dims], 
                  color=colors, edgecolor='black', linewidth=1.5)
    axes[1, 2].set_xticks(range(len(dims)))
    axes[1, 2].set_xticklabels(dims)
    axes[1, 2].set_ylabel('F1 Score', fontsize=11)
    axes[1, 2].set_title('F1 Score (Macro)', fontsize=13, fontweight='bold')
    axes[1, 2].grid(True, alpha=0.3, axis='y')
else:
    axes[1, 2].text(0.5, 0.5, 'No class labels available', 
                   ha='center', va='center', fontsize=14,
                   transform=axes[1, 2].transAxes)
    axes[1, 2].set_title('F1 Score (Macro)', fontsize=13, fontweight='bold')

for ax in axes.flat:
    ax.set_xlabel('Latent Dimension', fontsize=11)

plt.tight_layout()
plt.savefig(os.path.join(CONFIG['comparison_dir'], 'metrics_comparison.png'), 
            dpi=150, bbox_inches='tight')
plt.show()

print("✓ Metrics comparison charts generated")

## 8. Latent Space Visualization

### t-SNE Projections
Visualize the latent space structure using t-SNE dimensionality reduction

In [ ]:
# t-SNE Visualization colored by class
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('Latent Space Visualization (t-SNE) - Colored by Class', fontsize=18, fontweight='bold')

for idx, dim in enumerate(dims):
    r = evaluation_results[dim]
    
    if class_labels is not None:
        # Create color map
        unique_classes = np.unique(class_labels)
        n_classes = len(unique_classes)
        
        if n_classes <= 10:
            colors_map = plt.cm.tab10(np.linspace(0, 1, n_classes))
        else:
            colors_map = plt.cm.tab20(np.linspace(0, 1, min(20, n_classes)))
        
        class_to_color = dict(zip(unique_classes, colors_map))
        point_colors = [class_to_color[label] for label in class_labels]
        
        axes[idx].scatter(r['tsne_2d'][:, 0], r['tsne_2d'][:, 1], 
                         c=point_colors, s=2, alpha=0.6, edgecolors='none')
    else:
        axes[idx].scatter(r['tsne_2d'][:, 0], r['tsne_2d'][:, 1], 
                         c='steelblue', s=2, alpha=0.6, edgecolors='none')
    
    title = f'Latent Dim = {dim}'
    if r['silhouette'] is not None:
        title += f'\nSilhouette: {r["silhouette"]:.3f}'
    
    axes[idx].set_title(title, fontsize=14, fontweight='bold')
    axes[idx].set_xlabel('t-SNE Component 1', fontsize=12)
    axes[idx].set_ylabel('t-SNE Component 2', fontsize=12)
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(CONFIG['comparison_dir'], 'latent_space_tsne_by_class.png'), 
            dpi=150, bbox_inches='tight')
plt.show()

print("✓ t-SNE visualization (by class) generated")

In [ ]:
# t-SNE Visualization colored by reconstruction error
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('Latent Space Visualization (t-SNE) - Colored by Reconstruction Error', 
            fontsize=18, fontweight='bold')

for idx, dim in enumerate(dims):
    r = evaluation_results[dim]
    
    scatter = axes[idx].scatter(r['tsne_2d'][:, 0], r['tsne_2d'][:, 1], 
                               c=np.log10(r['reconstruction_errors'] + 1e-10), 
                               s=2, alpha=0.6, cmap='viridis', edgecolors='none')
    
    title = f'Latent Dim = {dim}\nMean Error: {r["mean_reconstruction_error"]:.6f}'
    axes[idx].set_title(title, fontsize=14, fontweight='bold')
    axes[idx].set_xlabel('t-SNE Component 1', fontsize=12)
    axes[idx].set_ylabel('t-SNE Component 2', fontsize=12)
    axes[idx].grid(True, alpha=0.3)
    
    cbar = plt.colorbar(scatter, ax=axes[idx])
    cbar.set_label('log₁₀(Reconstruction Error)', fontsize=11)

plt.tight_layout()
plt.savefig(os.path.join(CONFIG['comparison_dir'], 'latent_space_tsne_by_error.png'), 
            dpi=150, bbox_inches='tight')
plt.show()

print("✓ t-SNE visualization (by error) generated")

## 9. Reconstruction Quality Analysis

In [ ]:
# Visualize reconstructions for each model
fig, axes = plt.subplots(len(dims), 4, figsize=(20, 4*len(dims)))
fig.suptitle('Sample Reconstructions for All Models', fontsize=18, fontweight='bold')

# Select 4 random samples
sample_indices = np.random.choice(len(X_train), 4, replace=False)

for model_idx, dim in enumerate(dims):
    r = evaluation_results[dim]
    reconstructions = r['reconstructions']
    
    for sample_idx, idx in enumerate(sample_indices):
        ax = axes[model_idx, sample_idx] if len(dims) > 1 else axes[sample_idx]
        
        # Plot original and reconstruction
        ax.plot(X_train[idx], label='Original', linewidth=1.5, alpha=0.8)
        ax.plot(reconstructions[idx, 0, :], label='Reconstruction', 
               linewidth=1.5, alpha=0.8, linestyle='--')
        
        error = r['reconstruction_errors'][idx]
        title = f"Latent={dim}, Sample {idx}"
        if class_labels is not None:
            title += f"\nClass: {class_labels[idx]}"
        title += f"\nError: {error:.6f}"
        
        ax.set_title(title, fontsize=11)
        ax.set_xlabel('Index', fontsize=10)
        ax.set_ylabel('Intensity', fontsize=10)
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(CONFIG['comparison_dir'], 'reconstruction_samples.png'), 
            dpi=150, bbox_inches='tight')
plt.show()

print("✓ Reconstruction samples visualization generated")

In [ ]:
# Reconstruction error distribution
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Reconstruction Error Distribution', fontsize=16, fontweight='bold')

for idx, dim in enumerate(dims):
    r = evaluation_results[dim]
    
    axes[idx].hist(r['reconstruction_errors'], bins=50, 
                  color=colors[idx], alpha=0.7, edgecolor='black')
    axes[idx].axvline(r['mean_reconstruction_error'], 
                     color='red', linestyle='--', linewidth=2, 
                     label=f'Mean: {r["mean_reconstruction_error"]:.6f}')
    axes[idx].set_xlabel('Reconstruction Error (MSE)', fontsize=11)
    axes[idx].set_ylabel('Frequency', fontsize=11)
    axes[idx].set_title(f'Latent Dim = {dim}', fontsize=13, fontweight='bold')
    axes[idx].legend(fontsize=10)
    axes[idx].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(CONFIG['comparison_dir'], 'error_distribution.png'), 
            dpi=150, bbox_inches='tight')
plt.show()

print("✓ Error distribution visualization generated")

## 10. Summary and Conclusions

In [ ]:
# Save comprehensive results summary
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

results_summary = {
    'timestamp': timestamp,
    'config': CONFIG,
    'model_architecture': {
        'input_length': CONFIG['input_length'],
        'encoder_architecture': '1000 → 500 → 250 → 125 → 62 → 31 → latent_dim',
        'decoder_architecture': 'latent_dim → 31 → 62 → 125 → 250 → 500 → 1000',
    },
    'training_summary': {
        dim: {
            'total_training_time': trained_models[dim]['config']['total_time'],
            'final_loss': trained_models[dim]['history']['train_loss'][-1],
            'average_epoch_time': np.mean(trained_models[dim]['history']['epoch_times'])
        }
        for dim in CONFIG['latent_dims']
    },
    'evaluation_summary': {
        dim: {
            'reconstruction_error_mean': float(evaluation_results[dim]['mean_reconstruction_error']),
            'reconstruction_error_std': float(evaluation_results[dim]['std_reconstruction_error']),
            'nn_similarity_mean': float(evaluation_results[dim]['nn_similarity']),
            'nn_above_threshold': float(evaluation_results[dim]['nn_above_threshold']),
            'silhouette_score': float(evaluation_results[dim]['silhouette']) if evaluation_results[dim]['silhouette'] is not None else None,
            'f1_weighted': float(evaluation_results[dim]['f1_weighted']) if evaluation_results[dim]['f1_weighted'] is not None else None,
            'f1_macro': float(evaluation_results[dim]['f1_macro']) if evaluation_results[dim]['f1_macro'] is not None else None,
            'total_parameters': int(evaluation_results[dim]['total_params']),
            'compression_ratio': float(CONFIG['input_length'] / dim)
        }
        for dim in CONFIG['latent_dims']
    }
}

# Save to JSON
results_path = os.path.join(CONFIG['comparison_dir'], f'comprehensive_results_{timestamp}.json')
with open(results_path, 'w') as f:
    json.dump(results_summary, f, indent=2)

print(f"✓ Results saved to: {results_path}")

In [ ]:
# Print final summary
print("="*70)
print("COMPREHENSIVE MODEL COMPARISON SUMMARY")
print("="*70)
print(f"\nDataset: {CONFIG['data_path']}")
print(f"Number of samples: {len(X_train)}")
print(f"Input length: {CONFIG['input_length']}")
print(f"Training epochs: {CONFIG['epochs']}")
print(f"\n{'='*70}")
print(f"{'Metric':<30} {'Latent=128':<15} {'Latent=64':<15} {'Latent=32':<15}")
print("="*70)

# Reconstruction Error
print(f"{'Reconstruction Error':<30} ", end='')
for dim in CONFIG['latent_dims']:
    print(f"{evaluation_results[dim]['mean_reconstruction_error']:<15.6f}", end='')
print()

# NN Similarity
print(f"{'NN Similarity':<30} ", end='')
for dim in CONFIG['latent_dims']:
    print(f"{evaluation_results[dim]['nn_similarity']:<15.4f}", end='')
print()

# NN Above Threshold
print(f"{'NN Above 0.7 (%)':<30} ", end='')
for dim in CONFIG['latent_dims']:
    print(f"{evaluation_results[dim]['nn_above_threshold']:<15.2f}", end='')
print()

# Silhouette
if HAS_LABELS:
    print(f"{'Silhouette Score':<30} ", end='')
    for dim in CONFIG['latent_dims']:
        print(f"{evaluation_results[dim]['silhouette']:<15.4f}", end='')
    print()

# F1 Scores
if HAS_LABELS:
    print(f"{'F1 Score (weighted)':<30} ", end='')
    for dim in CONFIG['latent_dims']:
        print(f"{evaluation_results[dim]['f1_weighted']:<15.4f}", end='')
    print()
    
    print(f"{'F1 Score (macro)':<30} ", end='')
    for dim in CONFIG['latent_dims']:
        print(f"{evaluation_results[dim]['f1_macro']:<15.4f}", end='')
    print()

# Parameters
print(f"{'Total Parameters':<30} ", end='')
for dim in CONFIG['latent_dims']:
    print(f"{evaluation_results[dim]['total_params']:<15,}", end='')
print()

# Compression Ratio
print(f"{'Compression Ratio':<30} ", end='')
for dim in CONFIG['latent_dims']:
    print(f"{CONFIG['input_length']/dim:<15.1f}x", end='')
print()

print("="*70)

In [ ]:
# Key findings and recommendations
print("\n" + "="*70)
print("KEY FINDINGS AND RECOMMENDATIONS")
print("="*70)

# Find best model for each metric
best_recon = min(CONFIG['latent_dims'], 
                key=lambda d: evaluation_results[d]['mean_reconstruction_error'])
best_nn_sim = max(CONFIG['latent_dims'], 
                 key=lambda d: evaluation_results[d]['nn_similarity'])

print(f"\n1. Best Reconstruction Quality: Latent Dim = {best_recon}")
print(f"   → Lowest reconstruction error: {evaluation_results[best_recon]['mean_reconstruction_error']:.6f}")

print(f"\n2. Best Similarity Preservation: Latent Dim = {best_nn_sim}")
print(f"   → Highest NN similarity: {evaluation_results[best_nn_sim]['nn_similarity']:.4f}")

if HAS_LABELS:
    best_f1 = max(CONFIG['latent_dims'], 
                 key=lambda d: evaluation_results[d]['f1_weighted'])
    best_silhouette = max(CONFIG['latent_dims'], 
                         key=lambda d: evaluation_results[d]['silhouette'])
    
    print(f"\n3. Best Classification Performance: Latent Dim = {best_f1}")
    print(f"   → Highest F1 score: {evaluation_results[best_f1]['f1_weighted']:.4f}")
    
    print(f"\n4. Best Clustering Quality: Latent Dim = {best_silhouette}")
    print(f"   → Highest silhouette score: {evaluation_results[best_silhouette]['silhouette']:.4f}")

print(f"\n5. Model Complexity vs Performance:")
for dim in reversed(CONFIG['latent_dims']):
    params = evaluation_results[dim]['total_params']
    error = evaluation_results[dim]['mean_reconstruction_error']
    compression = CONFIG['input_length'] / dim
    print(f"   Latent={dim}: {params:,} params, {compression:.1f}x compression, {error:.6f} error")

print(f"\n{'='*70}")
print("RECOMMENDATION:")
print(f"{'='*70}")

# Simple recommendation logic
if best_recon == best_nn_sim:
    print(f"\n✓ Best overall model: Latent Dim = {best_recon}")
    print(f"  This model provides the best balance of reconstruction quality")
    print(f"  and similarity preservation.")
else:
    print(f"\n✓ For highest accuracy: Latent Dim = {best_recon}")
    print(f"✓ For best representations: Latent Dim = {best_nn_sim}")
    print(f"\n  Choose based on your application requirements:")
    print(f"  - Reconstruction tasks → Latent Dim = {best_recon}")
    print(f"  - Representation learning → Latent Dim = {best_nn_sim}")

print(f"\n{'='*70}")
print("Analysis complete! All results saved to './comparison_results/'")
print(f"{'='*70}")

## 📝 Conclusion

This notebook successfully demonstrated:

1. **Multi-Model Training**: Trained 3 TCN Autoencoder models with different latent dimensions
2. **Comprehensive Evaluation**: Evaluated using 5 different metrics
3. **Comparative Analysis**: Identified trade-offs between model complexity and performance
4. **Visualization**: Generated detailed visualizations of latent spaces and performance

### Next Steps
- Fine-tune hyperparameters for the best-performing model
- Apply the trained models to downstream tasks (classification, anomaly detection)
- Experiment with different architectures or training strategies
- Test on held-out validation/test data

### Files Generated
- Model checkpoints: `./outputs/tcn_autoencoder_latent*_epoch50_latest.pth`
- Comparison visualizations: `./comparison_results/*.png`
- Results summary: `./comparison_results/comprehensive_results_*.json`

---

**Author**: Generated for GitHub Repository  
**Date**: February 2026  
